# MuRIL — Hindi Fake News Detection

trying out google's MuRIL (Multilingual Representations for Indian Languages) for the CONSTRAINT 2021 hindi fake news dataset.
MuRIL was pretrained specifically on Indian languages + english, so should be a better baseline than vanilla mBERT for hindi text.

**model**: `google/muril-base-cased`  
**task**: binary clf — Fake (1) vs Real (0)  
**platform**: Kaggle T4 GPU  

plan: fine-tune the whole encoder with a standard CE head, early stop on macro-F1, then dump soft probs for later ensemble with Sarvam.


In [ ]:
# imports — keeping these together so i know what's being used
import os
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef,
    confusion_matrix, classification_report,
)

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# config — tuned these after a few runs
# ended up keeping lr at 2e-5, going lower didn't help much
# batch 16 because 32 kept OOM-ing on the validation step for some reason

MODEL_NAME = "google/muril-base-cased"
MAX_LEN    = 256   # 128 was losing context on longer posts
BATCH_SIZE = 16
EPOCHS     = 6
LR         = 2e-5
WARMUP     = 0.1   # warmup first 10% of steps, then linear decay
PATIENCE   = 2     # stop if val macro-F1 doesn't improve for 2 epochs

TRAIN_PATH = "/kaggle/input/datasets/sherwinmazarello/hindi-a/cleaned_file2.csv"
TEST_PATH  = "/kaggle/input/datasets/sherwinmazarello/hindi-a/cleaned_dataset_test.csv"
VALID_PATH = "/kaggle/input/datasets/sherwinmazarello/hindi-a/cleaned_dataset_valid.csv"

CKPT_PT  = "best_muril_model.pt"        # just saves the state dict, faster
CKPT_HF  = "best_muril_model_hf"        # full HF format for later loading
PROBS_DIR = "ensemble_probs"            # where soft probs get dumped for ensemble

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1. Load data + assign sample IDs

In [ ]:
# loading the 3 splits
# adding sample_id = original csv row index — need this later when
# aligning MuRIL and Sarvam predictions to the same articles in the ensemble
# (the two models might drop different rows during cleaning, so IDs let us intersect)

train_raw = pd.read_csv(TRAIN_PATH)
valid_raw = pd.read_csv(VALID_PATH)
test_raw  = pd.read_csv(TEST_PATH)

train_raw["sample_id"] = train_raw.index
valid_raw["sample_id"] = valid_raw.index
test_raw["sample_id"]  = test_raw.index

print("train:", train_raw.shape)
print("valid:", valid_raw.shape)
print("test :", test_raw.shape)


## 2. Label conversion & text cleaning

In [ ]:
# label: 1 = fake, 0 = non-fake
def convert_label(x):
    return 1 if "fake" in str(x).lower() else 0

# mild cleaning — don't want to strip too much hindi text
# just remove URLs and non-hindi/word chars, normalize whitespace
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^\w\sऀ-ॿ]", " ", text)   # keep devanagari range
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def prepare_df(df):
    df = df.copy()
    df["label"] = df["Labels Set"].apply(convert_label)
    df["text"]  = df["Post"].astype(str).apply(clean_text)
    df = df.drop_duplicates(subset="text")
    df = df[df["text"].str.split().str.len() > 5]   # drop super short posts
    return df[["text", "label", "sample_id"]].reset_index(drop=True)

train_df = prepare_df(train_raw)
valid_df = prepare_df(valid_raw)
test_df  = prepare_df(test_raw)

for name, df in [("Train", train_df), ("Valid", valid_df), ("Test", test_df)]:
    print(f"{name}: {len(df)} samples | fake={df['label'].sum()} real={(df['label']==0).sum()}")


## 3. Class weights  
Dataset is imbalanced (~20% fake), so using inverse-frequency weights in the loss.

In [ ]:
# inverse-frequency class weighting
# formula: total / (2 * class_count)  — keeps weights balanced around 1.0

n_total   = len(train_df)
n_fake    = int(train_df["label"].sum())
n_nonfake = n_total - n_fake

w_nonfake = n_total / (2 * n_nonfake)
w_fake    = n_total / (2 * n_fake)
class_weights = torch.tensor([w_nonfake, w_fake], dtype=torch.float)

print(f"non-fake: {n_nonfake}  weight={w_nonfake:.4f}")
print(f"fake    : {n_fake}     weight={w_fake:.4f}")


## 4. Tokeniser & Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("vocab size:", tokenizer.vocab_size)


In [ ]:
class FakeNewsDataset(Dataset):
    def __init__(self, df):
        self.texts      = df["text"].values
        self.labels     = df["label"].values
        self.sample_ids = df["sample_id"].values   # carrying IDs through for ensemble

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(int(self.labels[idx]),     dtype=torch.long),
            "sample_id":      torch.tensor(int(self.sample_ids[idx]), dtype=torch.long),
        }


In [ ]:
train_loader = DataLoader(FakeNewsDataset(train_df), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(FakeNewsDataset(valid_df), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(FakeNewsDataset(test_df),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"train batches: {len(train_loader)}")
print(f"valid batches: {len(valid_loader)}")
print(f"test  batches: {len(test_loader)}")


## 5. Model, optimiser, scheduler

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(WARMUP * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
loss_fn = CrossEntropyLoss(weight=class_weights.to(device))

print(f"total steps={total_steps}  warmup={warmup_steps}")
print(f"trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 6. Training loop

In [ ]:
best_val_f1      = 0.0
patience_counter = 0
train_f1_history = []
valid_f1_history = []

for epoch in range(EPOCHS):
    print(f"\nepoch {epoch+1}/{EPOCHS}")
    print("-" * 40)

    # ── train ──────────────────────────────────────────────────
    model.train()
    train_preds, train_targets = [], []

    for batch in train_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = loss_fn(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        train_targets.extend(lbls.cpu().numpy())

    train_f1 = f1_score(train_targets, train_preds, average="macro")
    train_f1_history.append(train_f1)
    print(f"  train macro-F1 : {train_f1:.4f}")

    # ── validate ────────────────────────────────────────────────
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch in valid_loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)
            preds = torch.argmax(
                model(input_ids=ids, attention_mask=mask).logits, 1
            ).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(lbls.cpu().numpy())

    val_f1 = f1_score(val_targets, val_preds, average="macro")
    valid_f1_history.append(val_f1)
    print(f"  val   macro-F1 : {val_f1:.4f}")

    # checkpoint on best val F1
    if val_f1 > best_val_f1:
        best_val_f1      = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), CKPT_PT)
        os.makedirs(CKPT_HF, exist_ok=True)
        model.save_pretrained(CKPT_HF)
        tokenizer.save_pretrained(CKPT_HF)
        print(f"  saved best model (val F1={best_val_f1:.4f})")
    else:
        patience_counter += 1
        print(f"  no improvement — patience {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("  early stopping.")
            break

print(f"\nbest val macro-F1: {best_val_f1:.4f}")


## 7. Evaluation on all splits  
Load best checkpoint, run on train/val/test.

In [ ]:
# reload best weights before evaluating
model.load_state_dict(torch.load(CKPT_PT, map_location=device))
model.eval()

def evaluate(loader, split_name):
    all_preds, all_labels, all_ids = [], [], []

    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)

            logits = model(input_ids=ids, attention_mask=mask).logits
            preds  = torch.argmax(logits, 1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(lbls.cpu().numpy())
            all_ids.extend(batch["sample_id"].numpy())

    acc      = accuracy_score(all_labels, all_preds)
    prec     = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec      = recall_score(all_labels, all_preds,    average="macro", zero_division=0)
    bin_f1   = f1_score(all_labels, all_preds, average="binary",  zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro",   zero_division=0)
    mcc      = matthews_corrcoef(all_labels, all_preds)

    print(f"\n{split_name}")
    print(f"  acc={acc:.4f}  prec={prec:.4f}  rec={rec:.4f}")
    print(f"  binary-F1={bin_f1:.4f}  macro-F1={macro_f1:.4f}  MCC={mcc:.4f}")
    print(classification_report(all_labels, all_preds,
                                target_names=["Non-Fake", "Fake"], digits=4))
    return all_preds, all_labels, all_ids


train_preds, train_labels, train_ids = evaluate(train_loader, "Train")
val_preds,   val_labels,   val_ids   = evaluate(valid_loader, "Validation")
test_preds,  test_labels,  test_ids  = evaluate(test_loader,  "Test")


## 8. Visualisations

In [ ]:
# confusion matrices for all 3 splits side by side
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, lbls, preds) in zip(axes, [
    ("Train",      train_labels, train_preds),
    ("Validation", val_labels,   val_preds),
    ("Test",       test_labels,  test_preds),
]):
    cm = confusion_matrix(lbls, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Non-Fake", "Fake"],
                yticklabels=["Non-Fake", "Fake"], cbar=False)
    ax.set_title(f"MuRIL — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# training curves — just F1 since that's the metric i was watching
epochs_ran = range(1, len(train_f1_history) + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_ran, train_f1_history, marker="o", label="Train Macro-F1")
plt.plot(epochs_ran, valid_f1_history, marker="s", label="Val Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("MuRIL — Train vs Val Macro-F1")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Error analysis  
Quick look at what the model is getting wrong on the test set.

In [ ]:
model.eval()
tp_texts, tn_texts, fp_texts, fn_texts = [], [], [], []

with torch.no_grad():
    for batch in test_loader:
        ids   = batch["input_ids"].to(device)
        mask  = batch["attention_mask"].to(device)
        lbls  = batch["labels"]
        preds = torch.argmax(
            model(input_ids=ids, attention_mask=mask).logits, 1
        ).cpu()

        for token_ids, pred, true in zip(ids, preds, lbls):
            text = tokenizer.decode(token_ids, skip_special_tokens=True)
            if len(text.split()) < 6:
                continue
            p, t = int(pred), int(true)
            if   t == 1 and p == 1 and len(tp_texts) < 3: tp_texts.append(text)
            elif t == 0 and p == 0 and len(tn_texts) < 3: tn_texts.append(text)
            elif t == 0 and p == 1 and len(fp_texts) < 3: fp_texts.append(text)
            elif t == 1 and p == 0 and len(fn_texts) < 3: fn_texts.append(text)

for category, samples in [
    ("TRUE POSITIVE  — caught fake correctly",   tp_texts),
    ("TRUE NEGATIVE  — real passed correctly",   tn_texts),
    ("FALSE POSITIVE — real flagged as fake",    fp_texts),
    ("FALSE NEGATIVE — fake missed as real",     fn_texts),
]:
    print(f"\n── {category}")
    for s in samples:
        print(f"   {s[:200]}")


## 10. Save soft probabilities + sample IDs for ensemble

In [ ]:
# saving softmax probs (shape N x 2) and the sample IDs
# so the ensemble notebook can do np.intersect1d(muril_ids, sarvam_ids)
# and only compare predictions on articles both models actually saw

def get_probs_and_ids(loader):
    model.eval()
    all_probs, all_ids = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            logits = model(input_ids=ids, attention_mask=mask).logits
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_ids.extend(batch["sample_id"].numpy())
    return np.array(all_probs), np.array(all_ids)


os.makedirs(PROBS_DIR, exist_ok=True)

train_probs, train_prob_ids = get_probs_and_ids(train_loader)
valid_probs, valid_prob_ids = get_probs_and_ids(valid_loader)
test_probs,  test_prob_ids  = get_probs_and_ids(test_loader)

np.save(f"{PROBS_DIR}/muril_train_probs.npy",  train_probs)
np.save(f"{PROBS_DIR}/muril_valid_probs.npy",  valid_probs)
np.save(f"{PROBS_DIR}/muril_test_probs.npy",   test_probs)
np.save(f"{PROBS_DIR}/train_labels.npy",        np.array(train_labels))
np.save(f"{PROBS_DIR}/valid_labels.npy",        np.array(val_labels))
np.save(f"{PROBS_DIR}/test_labels.npy",         np.array(test_labels))
np.save(f"{PROBS_DIR}/muril_train_ids.npy",    train_prob_ids)
np.save(f"{PROBS_DIR}/muril_valid_ids.npy",    valid_prob_ids)
np.save(f"{PROBS_DIR}/muril_test_ids.npy",     test_prob_ids)

print(f"saved to {PROBS_DIR}/")
print(f"  train probs: {train_probs.shape}  ids: {train_prob_ids.shape}")
print(f"  valid probs: {valid_probs.shape}  ids: {valid_prob_ids.shape}")
print(f"  test  probs: {test_probs.shape}   ids: {test_prob_ids.shape}")


## 11. Download model

In [ ]:
shutil.make_archive("best_muril_model_hf", "zip", CKPT_HF)
print("best_muril_model_hf.zip ready — download from Kaggle Output tab")
